# Teste Cerebras API — geração de exemplos N1

Este notebook valida o **Cerebras** como quinto provider ativo.

Fluxo:

1. ler `CEREBRAS_API_KEY` dos Secrets do Colab;
2. listar os modelos disponíveis;
3. selecionar automaticamente um modelo textual;
4. fazer o teste “Olá”;
5. gerar 2 exemplos N1;
6. validar o JSON;
7. verificar se a resposta aparece no contexto;
8. guardar e descarregar o resultado.

No Colab, cria o Secret:

`CEREBRAS_API_KEY`

In [48]:
# ============================================================
# CONFIGURAÇÃO
# ============================================================

NUMBER_OF_EXAMPLES = 2

OUTPUT_DIR = "/content/cerebras_outputs"
OUTPUT_FILE = (
    f"{OUTPUT_DIR}/n1_cerebras_test_"
    f"{NUMBER_OF_EXAMPLES}_examples.json"
)

DOMAIN = "Bibliotecas"
FACT_TYPE = "Capacidade"
ENTITY_TYPE = "Biblioteca"
CONTEXT_LENGTH = "curto, com 3 a 5 frases"
FACT_POSITION = "no fim do contexto"
QUESTION_TYPE = "pergunta direta"
DISTRACTOR_RULE = (
    "não incluir informação enganadora nem distratores relevantes"
)

# O notebook usa apenas os candidatos que estiverem disponíveis
# na tua conta, pela ordem definida abaixo.
MODEL_CANDIDATES = [
    "gemma-4-31b",
    "zai-glm-4.7",
    "gpt-oss-120b",
    "qwen-3-235b-a22b-instruct-2507",
]

TEMPERATURE = 0.8
MAX_TOKENS = 4000

print("Número de exemplos:", NUMBER_OF_EXAMPLES)
print("Ficheiro de saída:", OUTPUT_FILE)

Número de exemplos: 2
Ficheiro de saída: /content/cerebras_outputs/n1_cerebras_test_2_examples.json


In [49]:
# ============================================================
# IMPORTS E AUTENTICAÇÃO
# ============================================================

import json
import re
import time
import requests
from pathlib import Path
from google.colab import userdata

api_key = userdata.get("CEREBRAS_API_KEY")

if not api_key:
    raise ValueError(
        "O secret CEREBRAS_API_KEY não foi encontrado. "
        "Adiciona-o no painel Secrets do Colab."
    )

BASE_URL = "https://api.cerebras.ai/v1"
MODELS_URL = f"{BASE_URL}/models"
CHAT_URL = f"{BASE_URL}/chat/completions"

HEADERS = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json",
}

Path(OUTPUT_DIR).mkdir(
    parents=True,
    exist_ok=True,
)

print("Configuração pronta.")

Configuração pronta.


## Listar modelos disponíveis

In [50]:
# ============================================================
# LISTAR MODELOS
# ============================================================

models_response = requests.get(
    MODELS_URL,
    headers=HEADERS,
    timeout=60,
)

if not models_response.ok:
    print("Código HTTP:", models_response.status_code)
    print(models_response.text)

models_response.raise_for_status()

models_payload = models_response.json()

available_models = sorted(
    item["id"]
    for item in models_payload.get("data", [])
    if item.get("id")
)

print(f"Modelos encontrados: {len(available_models)}\n")

for model_id in available_models:
    print(model_id)

Modelos encontrados: 3

gemma-4-31b
gpt-oss-120b
zai-glm-4.7


## Selecionar automaticamente o modelo

In [51]:
# ============================================================
# SELEÇÃO DO MODELO
# ============================================================

MODEL_NAME = next(
    (
        candidate
        for candidate in MODEL_CANDIDATES
        if candidate in available_models
    ),
    None,
)

if MODEL_NAME is None:
    excluded_terms = [
        "embed",
        "rerank",
        "guard",
        "safety",
        "moderation",
        "ocr",
        "audio",
        "image",
    ]

    textual_models = [
        model_id
        for model_id in available_models
        if not any(
            term in model_id.lower()
            for term in excluded_terms
        )
    ]

    if not textual_models:
        raise RuntimeError(
            "Não foi encontrado um modelo textual disponível."
        )
    print(textual_models)
    MODEL_NAME = textual_models[0]

print("Modelo selecionado:", MODEL_NAME)

Modelo selecionado: gemma-4-31b


In [52]:
MODEL_CANDIDATES

['gemma-4-31b',
 'zai-glm-4.7',
 'gpt-oss-120b',
 'qwen-3-235b-a22b-instruct-2507']

## Função de chamada com repetição automática

In [53]:
# ============================================================
# CHAMADA HTTP COM RETRY
# ============================================================

RETRYABLE_CODES = {
    408, 409, 429, 500, 502, 503, 504
}


def extract_retry_after(response, default=10):
    header_value = response.headers.get("Retry-After")

    if header_value:
        try:
            return max(
                1,
                int(float(header_value)),
            )
        except ValueError:
            pass

    return default


def call_cerebras(
    messages,
    temperature,
    max_tokens,
    require_json=False,
    max_attempts=3,
):
    for attempt in range(1, max_attempts + 1):
        payload = {
            "model": MODEL_NAME,
            "messages": messages,
            "temperature": temperature,
            "max_tokens": max_tokens,
        }

        if require_json:
            payload["response_format"] = {
                "type": "json_object"
            }

        response = requests.post(
            CHAT_URL,
            headers=HEADERS,
            json=payload,
            timeout=180,
        )

        if response.ok:
            return response.json()

        print(
            f"Erro HTTP {response.status_code} "
            f"(tentativa {attempt}/{max_attempts})"
        )
        print(response.text[:1500])

        if (
            response.status_code in RETRYABLE_CODES
            and attempt < max_attempts
        ):
            wait_seconds = extract_retry_after(
                response
            )

            print(
                f"Aguardar {wait_seconds} segundos..."
            )

            time.sleep(wait_seconds + 1)
            continue

        response.raise_for_status()

    raise RuntimeError(
        "A Cerebras não respondeu após várias tentativas."
    )

## Teste simples da API

In [54]:
# ============================================================
# TESTE "OLÁ"
# ============================================================

test_data = call_cerebras(
    messages=[
        {
            "role": "user",
            "content": (
                "Responde apenas com a palavra: Olá"
            ),
        }
    ],
    temperature=0,
    max_tokens=20,
    require_json=False,
)

print(
    "Modelo reportado pela API:",
    test_data.get("model", MODEL_NAME),
)

print(
    "Resposta:",
    test_data["choices"][0]["message"]["content"],
)

Modelo reportado pela API: gemma-4-31b
Resposta: Olá


## Prompt fixa N1

Mantém o mesmo teste usado nos providers anteriores.

In [55]:
# ============================================================
# CONSTRUIR PROMPT
# ============================================================

prompt = f"""
Gera exatamente {NUMBER_OF_EXAMPLES} exemplos
em português europeu.

OBJETIVO
Criar exemplos para treinar um modelo pequeno
a localizar um facto explícito num contexto fornecido.

DISTRIBUIÇÃO
Todos os exemplos devem ter estas características:
- domínio: {DOMAIN}
- tipo de facto pedido: {FACT_TYPE}
- tipo de entidade principal: {ENTITY_TYPE}
- contexto: {CONTEXT_LENGTH}
- posição do facto que responde à pergunta:
  {FACT_POSITION}
- tipo de pergunta: {QUESTION_TYPE}
- distratores: {DISTRACTOR_RULE}

REGRAS
1. Cada exemplo deve conter exatamente:
   context, question e answer.
2. A resposta deve aparecer literalmente no contexto.
3. Deve existir apenas uma resposta correta.
4. A pergunta deve pedir apenas um facto.
5. Não uses conhecimento externo.
6. Usa entidades realistas,
   preferencialmente inventadas.
7. Não repitas exemplos.
8. Usa português europeu.
9. Devolve apenas JSON válido.
10. Não uses Markdown.
11. Não acrescentes explicações.

FORMATO OBRIGATÓRIO
{{
  "examples": [
    {{
      "context": "...",
      "question": "...",
      "answer": "..."
    }}
  ]
}}
""".strip()

print(prompt)

Gera exatamente 2 exemplos
em português europeu.

OBJETIVO
Criar exemplos para treinar um modelo pequeno
a localizar um facto explícito num contexto fornecido.

DISTRIBUIÇÃO
Todos os exemplos devem ter estas características:
- domínio: Bibliotecas
- tipo de facto pedido: Capacidade
- tipo de entidade principal: Biblioteca
- contexto: curto, com 3 a 5 frases
- posição do facto que responde à pergunta:
  no fim do contexto
- tipo de pergunta: pergunta direta
- distratores: não incluir informação enganadora nem distratores relevantes

REGRAS
1. Cada exemplo deve conter exatamente:
   context, question e answer.
2. A resposta deve aparecer literalmente no contexto.
3. Deve existir apenas uma resposta correta.
4. A pergunta deve pedir apenas um facto.
5. Não uses conhecimento externo.
6. Usa entidades realistas,
   preferencialmente inventadas.
7. Não repitas exemplos.
8. Usa português europeu.
9. Devolve apenas JSON válido.
10. Não uses Markdown.
11. Não acrescentes explicações.

FORMATO O

## Gerar os exemplos

In [56]:
# ============================================================
# GERAÇÃO
# ============================================================

generation_data = call_cerebras(
    messages=[
        {
            "role": "system",
            "content": (
                "És um gerador de dados estruturados. "
                "Devolve apenas JSON válido."
            ),
        },
        {
            "role": "user",
            "content": prompt,
        },
    ],
    temperature=TEMPERATURE,
    max_tokens=MAX_TOKENS,
    require_json=True,
)

actual_model = generation_data.get(
    "model",
    MODEL_NAME,
)

raw_text = (
    generation_data["choices"][0]
    ["message"]["content"]
    .strip()
)

print("Modelo reportado pela API:", actual_model)
print("\nResposta:")
print(raw_text[:2000])

Modelo reportado pela API: gemma-4-31b

Resposta:
{"examples": [{"context": "A Biblioteca Municipal de Alvor é um espaço moderno e acolhedor. Possui diversas secções dedicadas à literatura juvenil e história local. O edifício tem capacidade para 150 utilizadores.", "question": "Qual é a capacidade da Biblioteca Municipal de Alvor?", "answer": "150 utilizadores"}, {"context": "A Biblioteca Central de Mirandela oferece acesso a vastos arquivos digitais. O recinto conta com salas de estudo individuais e áreas de leitura. A instalação comporta 300 leitores simultaneamente.", "question": "Quantos leitores comporta a Biblioteca Central de Mirandela?", "answer": "300 leitores simultaneamente"}]}


## Validar o JSON

In [57]:
# ============================================================
# VALIDAÇÃO
# ============================================================

def clean_json_text(text):
    text = text.strip()

    if text.startswith("```json"):
        text = text[len("```json"):]

    if text.startswith("```"):
        text = text[len("```"):]

    if text.endswith("```"):
        text = text[:-3]

    return text.strip()


clean_text = clean_json_text(raw_text)

try:
    payload = json.loads(clean_text)
except json.JSONDecodeError as exc:
    print(clean_text)

    raise ValueError(
        f"A Cerebras não devolveu JSON válido: {exc}"
    ) from exc

if not isinstance(payload, dict):
    raise TypeError(
        "O resultado principal deve ser um objeto JSON."
    )

if "examples" not in payload:
    raise ValueError(
        "O JSON não contém a chave 'examples'."
    )

examples = payload["examples"]

if not isinstance(examples, list):
    raise TypeError(
        "'examples' deve ser um array JSON."
    )

if len(examples) != NUMBER_OF_EXAMPLES:
    raise ValueError(
        f"Esperava {NUMBER_OF_EXAMPLES} exemplos, "
        f"mas recebi {len(examples)}."
    )

required_fields = {
    "context",
    "question",
    "answer",
}

for index, example in enumerate(
    examples,
    start=1,
):
    if not isinstance(example, dict):
        raise TypeError(
            f"O exemplo {index} não é um objeto JSON."
        )

    missing = (
        required_fields
        - set(example.keys())
    )

    if missing:
        raise ValueError(
            f"Exemplo {index}: campos em falta: "
            f"{sorted(missing)}"
        )

    for field in required_fields:
        if (
            not isinstance(
                example[field],
                str,
            )
            or not example[field].strip()
        ):
            raise ValueError(
                f"Exemplo {index}: "
                f"campo '{field}' inválido."
            )

print(
    f"JSON válido: {len(examples)} exemplos."
)

JSON válido: 2 exemplos.


## Verificar as respostas

In [58]:
# ============================================================
# VERIFICAÇÃO BÁSICA
# ============================================================

def normalize(text):
    text = str(text).lower().strip()
    text = re.sub(r"\s+", " ", text)
    return text.strip(" .,:;!?")


all_valid = True

for index, example in enumerate(
    examples,
    start=1,
):
    answer_in_context = (
        normalize(example["answer"])
        in normalize(example["context"])
    )

    print(
        f"Exemplo {index}: "
        f"resposta no contexto = "
        f"{answer_in_context}"
    )

    if not answer_in_context:
        all_valid = False

if all_valid:
    print(
        "\nTodas as respostas aparecem "
        "literalmente nos contextos."
    )
else:
    print(
        "\nExistem exemplos que necessitam "
        "de revisão."
    )

Exemplo 1: resposta no contexto = True
Exemplo 2: resposta no contexto = True

Todas as respostas aparecem literalmente nos contextos.


## Mostrar os exemplos

In [59]:
# ============================================================
# APRESENTAÇÃO
# ============================================================

for index, example in enumerate(
    examples,
    start=1,
):
    print("\n" + "=" * 90)
    print(f"EXEMPLO {index}")
    print("=" * 90)

    print("CONTEXTO:")
    print(example["context"])

    print("\nPERGUNTA:")
    print(example["question"])

    print("\nRESPOSTA:")
    print(example["answer"])


EXEMPLO 1
CONTEXTO:
A Biblioteca Municipal de Alvor é um espaço moderno e acolhedor. Possui diversas secções dedicadas à literatura juvenil e história local. O edifício tem capacidade para 150 utilizadores.

PERGUNTA:
Qual é a capacidade da Biblioteca Municipal de Alvor?

RESPOSTA:
150 utilizadores

EXEMPLO 2
CONTEXTO:
A Biblioteca Central de Mirandela oferece acesso a vastos arquivos digitais. O recinto conta com salas de estudo individuais e áreas de leitura. A instalação comporta 300 leitores simultaneamente.

PERGUNTA:
Quantos leitores comporta a Biblioteca Central de Mirandela?

RESPOSTA:
300 leitores simultaneamente


## Guardar o resultado

In [60]:
# ============================================================
# GUARDAR JSON
# ============================================================

result = {
    "provider": "cerebras",
    "secret_name": "CEREBRAS_API_KEY",
    "requested_model": MODEL_NAME,
    "actual_model": actual_model,
    "number_of_examples": len(examples),
    "generation_settings": {
        "temperature": TEMPERATURE,
        "max_tokens": MAX_TOKENS,
        "domain": DOMAIN,
        "fact_type": FACT_TYPE,
        "entity_type": ENTITY_TYPE,
        "context_length": CONTEXT_LENGTH,
        "fact_position": FACT_POSITION,
        "question_type": QUESTION_TYPE,
    },
    "usage": generation_data.get(
        "usage",
        {},
    ),
    "examples": examples,
}

with open(
    OUTPUT_FILE,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        result,
        file,
        ensure_ascii=False,
        indent=2,
    )

print("Guardado em:")
print(OUTPUT_FILE)

Guardado em:
/content/cerebras_outputs/n1_cerebras_test_2_examples.json


## Descarregar

In [61]:
# ============================================================
# DOWNLOAD
# ============================================================

from google.colab import files

files.download(OUTPUT_FILE)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Estado dos providers

- Gemini: validado
- Groq: validado
- Mistral: validado
- OpenRouter Free: validado
- Cerebras: em validação
- xAI/Grok: retirado do conjunto ativo por ausência de créditos